# MarketLab BEAST Mode - Day 1 - Batch 1
   Processes stocks 1-17 with 30 models + optimization
   

In [ ]:
!nvidia-smi

Mon Apr 13 05:45:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q lightgbm xgboost catboost
print("✅ Runtime → Restart → Run CELL 2")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 28.5 MB/s eta 0:00:00
✅ Runtime → Restart → Run CELL 2


In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor, RANSACRegressor, TheilSenRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, AdaBoostRegressor, BaggingRegressor, VotingRegressor, StackingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

print("🔥 MARKETLAB BEAST - BATCH 1 - OPTIMIZED (25 MODELS × 325 FEATURES)\n" + "="*80)

BATCH_1_STOCKS = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS', 'HINDUNILVR.NS', 'ITC.NS', 'SBIN.NS', 'BHARTIARTL.NS', 'KOTAKBANK.NS', 'LT.NS', 'AXISBANK.NS', 'ASIANPAINT.NS', 'MARUTI.NS', 'HCLTECH.NS', 'SUNPHARMA.NS', 'TITAN.NS']

def load_csv(ticker):
    path = f"/content/drive/MyDrive/MarketLab_BEAST/stock_data/{ticker}.csv"
    with open(path, 'r') as f:
        first_line = f.readline()
    if 'Price","Open' in first_line:
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
        df = df.set_index('Date')
        df = df.rename(columns={'Price': 'Close', 'Vol.': 'Volume'})
        if df['Volume'].dtype == 'object':
            df['Volume'] = df['Volume'].str.replace('M', '').str.replace('K', '').str.replace(',', '')
            df['Volume'] = df['Volume'].apply(lambda x: float(x)*1000000 if 'M' in str(x) else (float(x)*1000 if 'K' in str(x) else float(x) if x else 0))
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    else:
        df = pd.read_csv(path, skiprows=[1])
        df = df.iloc[1:]
        df['Date'] = pd.to_datetime(df['Price'])
        df = df.set_index('Date')
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna().sort_index()

def generate_325_features(df):
    df['hl_ratio']=df['High']/df['Low']
    df['co_ratio']=df['Close']/df['Open']
    df['oc_diff']=df['Open']-df['Close']
    df['hl_diff']=df['High']-df['Low']
    df['body_size']=abs(df['Close']-df['Open'])
    df['upper_shadow']=df['High']-df[['Open','Close']].max(axis=1)
    df['lower_shadow']=df[['Open','Close']].min(axis=1)-df['Low']
    df['body_position']=(df['Close']-df['Low'])/(df['High']-df['Low'])
    for p in [1,2,3,5,7,10,14,21,30,60,90,120,180,252]:
        df[f'ret_{p}']=df['Close'].pct_change(p)
        df[f'log_ret_{p}']=np.log(df['Close']/df['Close'].shift(p))
    for p in [3,5,7,10,14,20,30,50,100,150,200]:
        df[f'sma_{p}']=df['Close'].rolling(p).mean()
        df[f'ema_{p}']=df['Close'].ewm(span=p).mean()
        df[f'close_sma_ratio_{p}']=df['Close']/df[f'sma_{p}']
        df[f'close_ema_ratio_{p}']=df['Close']/df[f'ema_{p}']
    for p in [5,10,20,30,60,90]:
        df[f'volatility_{p}']=df['Close'].pct_change().rolling(p).std()
        df[f'volatility_{p}_annual']=df[f'volatility_{p}']*np.sqrt(252)
        df[f'parkinson_{p}']=np.sqrt((1/(4*np.log(2)))*((np.log(df['High']/df['Low']))**2).rolling(p).mean())
        df[f'garman_klass_{p}']=np.sqrt(0.5*((np.log(df['High']/df['Low']))**2).rolling(p).mean()-(2*np.log(2)-1)*((np.log(df['Close']/df['Open']))**2).rolling(p).mean())
    for p in [7,9,14,21,28]:
        delta=df['Close'].diff()
        gain=delta.where(delta>0,0).rolling(p).mean()
        loss=-delta.where(delta<0,0).rolling(p).mean()
        rs=gain/loss
        df[f'rsi_{p}']=100-(100/(1+rs))
        df[f'rsi_{p}_slope']=df[f'rsi_{p}'].diff(5)
        df[f'rsi_{p}_ema']=df[f'rsi_{p}'].ewm(span=5).mean()
    for fast,slow,signal in [(12,26,9),(5,35,5),(19,39,9)]:
        ema_fast=df['Close'].ewm(span=fast).mean()
        ema_slow=df['Close'].ewm(span=slow).mean()
        df[f'macd_{fast}_{slow}']=ema_fast-ema_slow
        df[f'macd_signal_{fast}_{slow}_{signal}']=df[f'macd_{fast}_{slow}'].ewm(span=signal).mean()
        df[f'macd_hist_{fast}_{slow}_{signal}']=df[f'macd_{fast}_{slow}']-df[f'macd_signal_{fast}_{slow}_{signal}']
        df[f'macd_slope_{fast}_{slow}']=df[f'macd_{fast}_{slow}'].diff(3)
    for p in [10,20,30,50]:
        for std_mult in [1.5,2.0,2.5]:
            sma=df['Close'].rolling(p).mean()
            std=df['Close'].rolling(p).std()
            df[f'bb_upper_{p}_{std_mult}']=sma+(std_mult*std)
            df[f'bb_lower_{p}_{std_mult}']=sma-(std_mult*std)
            df[f'bb_width_{p}_{std_mult}']=(df[f'bb_upper_{p}_{std_mult}']-df[f'bb_lower_{p}_{std_mult}'])/sma
            df[f'bb_position_{p}_{std_mult}']=(df['Close']-df[f'bb_lower_{p}_{std_mult}'])/(df[f'bb_upper_{p}_{std_mult}']-df[f'bb_lower_{p}_{std_mult}'])
    for p in [5,10,20,30,50]:
        df[f'volume_sma_{p}']=df['Volume'].rolling(p).mean()
        df[f'volume_ratio_{p}']=df['Volume']/df[f'volume_sma_{p}']
        df[f'volume_std_{p}']=df['Volume'].rolling(p).std()
        df[f'volume_cv_{p}']=df[f'volume_std_{p}']/df[f'volume_sma_{p}']
        df[f'obv_{p}']=(np.sign(df['Close'].diff())*df['Volume']).rolling(p).sum()
        df[f'vwap_{p}']=(df['Close']*df['Volume']).rolling(p).sum()/df['Volume'].rolling(p).sum()
    for p in [5,10,20,30,50]:
        df[f'momentum_{p}']=df['Close']-df['Close'].shift(p)
        df[f'roc_{p}']=df['Close'].pct_change(p)*100
        df[f'trix_{p}']=df['Close'].ewm(span=p).mean().ewm(span=p).mean().ewm(span=p).mean().pct_change()
    for p in [7,14,21,30]:
        high_low=df['High']-df['Low']
        high_close=np.abs(df['High']-df['Close'].shift())
        low_close=np.abs(df['Low']-df['Close'].shift())
        ranges=pd.concat([high_low,high_close,low_close],axis=1)
        true_range=ranges.max(axis=1)
        df[f'atr_{p}']=true_range.rolling(p).mean()
        df[f'atr_ratio_{p}']=df[f'atr_{p}']/df['Close']
        df[f'natr_{p}']=(df[f'atr_{p}']/df['Close'])*100
    for p in [10,20,30,50]:
        df[f'std_{p}']=df['Close'].rolling(p).std()
        df[f'skew_{p}']=df['Close'].rolling(p).skew()
        df[f'kurt_{p}']=df['Close'].rolling(p).kurt()
        df[f'median_{p}']=df['Close'].rolling(p).median()
        df[f'quantile_25_{p}']=df['Close'].rolling(p).quantile(0.25)
        df[f'quantile_75_{p}']=df['Close'].rolling(p).quantile(0.75)
        df[f'iqr_{p}']=df[f'quantile_75_{p}']-df[f'quantile_25_{p}']
    for lag in [1,2,3,5,7,10,14,21,30]:
        df[f'close_lag_{lag}']=df['Close'].shift(lag)
        df[f'volume_lag_{lag}']=df['Volume'].shift(lag)
        df[f'high_lag_{lag}']=df['High'].shift(lag)
        df[f'low_lag_{lag}']=df['Low'].shift(lag)
    for p in [10,20,50,100]:
        df[f'high_max_{p}']=df['High'].rolling(p).max()
        df[f'low_min_{p}']=df['Low'].rolling(p).min()
        df[f'distance_high_{p}']=(df['Close']-df[f'high_max_{p}'])/df['Close']
        df[f'distance_low_{p}']=(df['Close']-df[f'low_min_{p}'])/df['Close']
    for p in [14,21]:
        low_min=df['Low'].rolling(p).min()
        high_max=df['High'].rolling(p).max()
        df[f'stoch_k_{p}']=100*(df['Close']-low_min)/(high_max-low_min)
        df[f'stoch_d_{p}']=df[f'stoch_k_{p}'].rolling(3).mean()
    high_9=df['High'].rolling(9).max()
    low_9=df['Low'].rolling(9).min()
    df['tenkan_sen']=(high_9+low_9)/2
    high_26=df['High'].rolling(26).max()
    low_26=df['Low'].rolling(26).min()
    df['kijun_sen']=(high_26+low_26)/2
    df['senkou_span_a']=((df['tenkan_sen']+df['kijun_sen'])/2).shift(26)
    high_52=df['High'].rolling(52).max()
    low_52=df['Low'].rolling(52).min()
    df['senkou_span_b']=((high_52+low_52)/2).shift(26)
    df['chikou_span']=df['Close'].shift(-26)
    return df

def get_25_models():
    """25 optimized models - removed 5 slowest (SVR_R, SVR_P, SVR_L, SVR_S, GP)"""
    m={}
    m['LR']=LinearRegression()
    m['Ridge']=Ridge()
    m['Lasso']=Lasso(max_iter=2000)
    m['EN']=ElasticNet(max_iter=2000)
    m['Bay']=BayesianRidge()
    m['Huber']=HuberRegressor()
    m['RF']=RandomForestRegressor(n_estimators=100,max_depth=10,n_jobs=-1,random_state=42)
    m['ET']=ExtraTreesRegressor(n_estimators=100,max_depth=10,n_jobs=-1,random_state=42)
    m['GB']=GradientBoostingRegressor(n_estimators=100,random_state=42)
    m['HGB']=HistGradientBoostingRegressor(random_state=42)
    m['XGB']=XGBRegressor(n_estimators=100,n_jobs=-1,random_state=42,verbosity=0)
    m['LGBM']=LGBMRegressor(n_estimators=100,n_jobs=-1,random_state=42,verbosity=-1)
    m['CB']=CatBoostRegressor(iterations=100,random_state=42,verbose=0)
    m['RFD']=RandomForestRegressor(n_estimators=100,max_depth=20,n_jobs=-1,random_state=42)
    m['Ada']=AdaBoostRegressor(n_estimators=50,random_state=42)
    m['Ada_L']=AdaBoostRegressor(estimator=LinearRegression(),n_estimators=50,random_state=42)
    m['GB_H']=GradientBoostingRegressor(n_estimators=100,loss='huber',random_state=42)
    m['KR']=KernelRidge()
    m['KNN']=KNeighborsRegressor(n_neighbors=5,n_jobs=-1)
    m['Bag']=BaggingRegressor(estimator=LinearRegression(),n_estimators=10,n_jobs=-1,random_state=42)
    e=[('r',Ridge()),('rf',RandomForestRegressor(n_estimators=50,n_jobs=-1,random_state=42))]
    m['Vote']=VotingRegressor(estimators=e)
    m['Vote_W']=VotingRegressor(estimators=e,weights=[2,3])
    e2=[('r',Ridge()),('rf',RandomForestRegressor(n_estimators=50,n_jobs=-1,random_state=42)),('gb',GradientBoostingRegressor(n_estimators=50,random_state=42))]
    m['Stack']=StackingRegressor(estimators=e2,final_estimator=Ridge(),n_jobs=-1)
    m['RANSAC']=RANSACRegressor(random_state=42,max_trials=100)
    m['TS']=TheilSenRegressor(random_state=42,max_iter=300,n_jobs=-1)
    return m

def process_stock(ticker):
    print(f"\n{'='*80}\n📊 {ticker}\n{'='*80}")
    try:
        print("   📥 Loading...")
        df=load_csv(ticker)
        print(f"      ✅ {len(df)} rows")
        print("   🔧 Generating 325 features...")
        df=generate_325_features(df)
        df['target']=df['Close'].shift(-1)
        df=df.dropna()
        ex=['Open','High','Low','Close','Volume','target']
        feat=[c for c in df.columns if c not in ex]
        X,y=df[feat],df['target']
        s=int(0.8*len(X))
        Xtr,Xte,ytr,yte=X.iloc[:s],X.iloc[s:],y.iloc[:s],y.iloc[s:]
        print(f"      ✅ {len(feat)} features, train={len(Xtr)}, test={len(Xte)}")
        print("   🤖 Training 25 models...")
        models=get_25_models()
        res={}
        for name,model in models.items():
            try:
                model.fit(Xtr,ytr)
                p=model.predict(Xte)
                r2=r2_score(yte,p)
                rmse=np.sqrt(mean_squared_error(yte,p))
                res[name]={'r2':r2,'mae':mean_absolute_error(yte,p),'rmse':rmse}
                print(f"      ✅ {name:15s} R²={r2:.4f}")
            except:
                pass
        out=Path('/content/drive/MyDrive/MarketLab_BEAST/results_day1')
        out.mkdir(exist_ok=True,parents=True)
        clean=ticker.replace('.NS','')
        ts=datetime.now().strftime("%Y%m%d_%H%M%S")
        pd.DataFrame(res).T.to_csv(out/f"{clean}_{ts}.csv")
        best=max([v['r2'] for v in res.values()])
        print(f"   ✅ DONE! Best R²={best:.4f}")
        return {'ticker':ticker,'best_r2':best,'models':len(res),'features':len(feat)}
    except Exception as e:
        print(f"   ❌ ERROR: {e}")
        return None

print("\n🚀 BATCH 1 (17 stocks)...\n")
results=[]
for i,t in enumerate(BATCH_1_STOCKS,1):
    print(f"Stock {i}/17")
    r=process_stock(t)
    if r:results.append(r)
print("\n"+"="*80)
print(f"🎉 COMPLETE! {len(results)}/17")
if results:
    s=pd.DataFrame(results)
    print(f"Features:{s['features'].iloc[0]} | Models:25 | Avg R²:{s['best_r2'].mean():.4f}")
print("="*80)

🔥 MARKETLAB BEAST - BATCH 1 - OPTIMIZED (25 MODELS × 325 FEATURES)

🚀 BATCH 1 (17 stocks)...

Stock 1/17

📊 RELIANCE.NS
   📥 Loading...
      ✅ 5193 rows
   🔧 Generating 325 features...
      ✅ 325 features, train=3926, test=982
   🤖 Training 25 models...
      ✅ LR              R²=0.9882
      ✅ Ridge           R²=0.9892
      ✅ Lasso           R²=0.9901
      ✅ EN              R²=0.9901
      ✅ Bay             R²=0.9900
      ✅ Huber           R²=-37.1507
      ✅ RF              R²=-2.5738
      ✅ ET              R²=-1.5555
      ✅ GB              R²=-1.8268
      ✅ HGB             R²=-2.5769
      ✅ XGB             R²=-2.4592
      ✅ LGBM            R²=-2.4560
      ✅ CB              R²=-2.8053
      ✅ RFD             R²=-2.5481
      ✅ Ada             R²=-2.3762
      ✅ Ada_L           R²=0.9174
      ✅ GB_H            R²=-1.6796
      ✅ KR              R²=0.9893
      ✅ KNN             R²=-27.4723
      ✅ Bag             R²=0.9885
      ✅ Vote            R²=0.0861
      ✅ Vote_W  